In [17]:
import os
import re
import unicodedata

import pandas as pd
from google_play_scraper import Sort, reviews

In [18]:
target_apps = [
    ("com.nationstrust.frimi", "FriMi_FinTech"),
    ("com.jkit.keellsretailapp", "Keells_Supermarket"),
    ("com.pb.peoplespayrc", "Peoples_Pay"),
    ("combank.com.combankdigital", "ComBank_Digital"),
    ("com.bankofceylon.flex", "FlexPay_BOC"),
    ("lk.nsbpay.user", "NSBPay"),
    ("com.hnb.DigitalApp", "HNB_DigitalBanking"),
    ("com.ndb.mobilebanking", "NDB_MobileBanking")
]

MIN_WORD_COUNT = 5

RAW_DIR = r"C:\Users\amala\Documents\projects\finsentiment2 - Copy2\data\raw"
PROCESSED_DIR = r"C:\Users\amala\Documents\projects\finsentiment2 - Copy2\data\processed"
COMBINED_FILE = os.path.join(PROCESSED_DIR, "scraped_reviews_combined.csv")

In [19]:
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def is_valid_english_text(text):
    if not text or pd.isna(text):
        return False
    clean_text = re.sub(r'[^a-zA-Z\s]', '', str(text))
    return len(clean_text.strip()) > 0

In [20]:
def scrape_app_reviews(app_id, app_name_label):
    print(f"\n Scraping started for: {app_name_label} ({app_id})...")

    all_scraped_reviews = []
    continuation_token = None
    total_raw_scanned = 0

    while True:
        try:
            result, continuation_token = reviews(
                app_id,
                lang='en',
                country='lk',
                sort=Sort.NEWEST,
                count=200,
                continuation_token=continuation_token
            )

            if not result:
                break

            total_raw_scanned += len(result)

            for item in result:
                text_content = item.get("content", "")
                word_list = str(text_content).split()

                if len(word_list) >= MIN_WORD_COUNT and is_valid_english_text(text_content):
                    all_scraped_reviews.append({
                        "source_app": app_name_label,
                        "review_text": text_content,
                        "star_rating": item.get("score"),
                        "recorded_at": item.get("at"),
                        "app_version": item.get("reviewCreatedVersion")
                    })

            if continuation_token is None:
                break

        except Exception as e:
            print(f"   Error fetching batch for {app_name_label}: {e}")
            break

    print(f" Finished {app_name_label}: scanned={total_raw_scanned}, valid={len(all_scraped_reviews)}")
    return all_scraped_reviews

In [21]:
def append_new_reviews(app_name_label, reviews_data):
    output_path = os.path.join(RAW_DIR, f"{app_name_label}_raw_reviews.csv")
    new_df = pd.DataFrame(reviews_data)

    if new_df.empty:
        return 0

    new_df["normalized_text"] = new_df["review_text"].apply(normalize_text)
    new_df.drop_duplicates(subset=["normalized_text", "recorded_at"], keep="first", inplace=True)

    if os.path.exists(output_path):
        existing_df = pd.read_csv(output_path)
        if "normalized_text" not in existing_df.columns:
            existing_df["normalized_text"] = existing_df["review_text"].apply(normalize_text)
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
        combined_df.drop_duplicates(subset=["normalized_text", "recorded_at"], keep="first", inplace=True)
    else:
        combined_df = new_df

    combined_df.reset_index(drop=True, inplace=True)
    combined_df["source_row_id"] = range(1, len(combined_df) + 1)
    os.makedirs(RAW_DIR, exist_ok=True)
    combined_df.to_csv(output_path, index=False, encoding="utf-8-sig")

    print(f" SAVED: {output_path} | Total rows: {len(combined_df)}")
    return len(combined_df)

In [22]:
for app_id, app_name_label in target_apps:
    reviews_data = scrape_app_reviews(app_id, app_name_label)

    if reviews_data:
        append_new_reviews(app_name_label, reviews_data)
    else:
        print(f" No valid data collected for {app_name_label}")



 Scraping started for: FriMi_FinTech (com.nationstrust.frimi)...
 Finished FriMi_FinTech: scanned=3445, valid=2235
 SAVED: C:\Users\amala\Documents\projects\finsentiment2 - Copy2\data\raw\FriMi_FinTech_raw_reviews.csv | Total rows: 6555

 Scraping started for: Keells_Supermarket (com.jkit.keellsretailapp)...
 Finished Keells_Supermarket: scanned=446, valid=326
 SAVED: C:\Users\amala\Documents\projects\finsentiment2 - Copy2\data\raw\Keells_Supermarket_raw_reviews.csv | Total rows: 960

 Scraping started for: Peoples_Pay (com.pb.peoplespayrc)...
 Finished Peoples_Pay: scanned=4278, valid=1139
 SAVED: C:\Users\amala\Documents\projects\finsentiment2 - Copy2\data\raw\Peoples_Pay_raw_reviews.csv | Total rows: 3128

 Scraping started for: ComBank_Digital (combank.com.combankdigital)...
 Finished ComBank_Digital: scanned=6290, valid=3631
 SAVED: C:\Users\amala\Documents\projects\finsentiment2 - Copy2\data\raw\ComBank_Digital_raw_reviews.csv | Total rows: 10068

 Scraping started for: FlexPay_

In [23]:
def combine_app_files_to_master(raw_dir, combined_file):
    csv_files = [
        f for f in os.listdir(raw_dir)
        if f.endswith("_raw_reviews.csv")
    ]

    if not csv_files:
        print("No app files found to combine.")
        return 0

    all_new_parts = []

    for file_name in csv_files:
        file_path = os.path.join(raw_dir, file_name)
        df = pd.read_csv(file_path)

        if df.empty:
            continue

        if "normalized_text" not in df.columns:
            df["normalized_text"] = df["review_text"].apply(normalize_text)

        all_new_parts.append(df)

    if not all_new_parts:
        print("No data found in app files.")
        return 0

    # Combine all app files
    new_data = pd.concat(all_new_parts, ignore_index=True)

    # Remove duplicates among app files
    new_data.drop_duplicates(
        subset=["normalized_text", "recorded_at"],
        keep="first",
        inplace=True
    )

    if os.path.exists(combined_file):

        existing = pd.read_csv(combined_file)

        if "normalized_text" not in existing.columns:
            existing["normalized_text"] = existing["review_text"].apply(normalize_text)

        before_count = len(existing)

        # Merge existing + new
        merged = pd.concat([existing, new_data], ignore_index=True)

        # Remove duplicates
        merged.drop_duplicates(
            subset=["normalized_text", "recorded_at"],
            keep="first",
            inplace=True
        )

        # Reset index
        merged.reset_index(drop=True, inplace=True)

        # Reassign source_row_id
        merged["source_row_id"] = merged.index + 1

        # Save
        merged.to_csv(combined_file, index=False, encoding="utf-8-sig")

        added_rows = len(merged) - before_count

        print(f" UPDATED: {combined_file}")
        print(f"New rows added : {added_rows}")
        print(f"Total rows     : {len(merged)}")

        return added_rows


    else:

        new_data.reset_index(drop=True, inplace=True)
        new_data["source_row_id"] = new_data.index + 1

        new_data.to_csv(combined_file, index=False, encoding="utf-8-sig")

        print(f" CREATED: {combined_file}")
        print(f"Total rows : {len(new_data)}")

        return len(new_data)

In [24]:
new_rows_added = combine_app_files_to_master(RAW_DIR, COMBINED_FILE)
print("New rows added to combined file:", new_rows_added)

 UPDATED: C:\Users\amala\Documents\projects\finsentiment2 - Copy2\data\processed\scraped_reviews_combined.csv
New rows added : 0
Total rows     : 23727
New rows added to combined file: 0


In [25]:
if os.path.exists(COMBINED_FILE):
    df_combined = pd.read_csv(COMBINED_FILE)
    print("Total rows in combined file:", len(df_combined))
else:
    print("Combined file does not exist yet.")

Total rows in combined file: 23727


In [26]:
if "df_combined" in globals():
    print(df_combined["star_rating"].unique())


[1 2 4 5 3]


In [27]:
# Create a function that maps star ratings to clean text categories
def assign_sentiment_text(stars):
    stars = int(stars)
    if stars in [1, 2]:
        return "negative"
    elif stars == 3:
        return "neutral"
    else:  # 4 and 5 stars
        return "positive"

# Apply the text categorization rule to your scraped dataset
df_combined["sentiment_text"] = df_combined["star_rating"].apply(assign_sentiment_text)

# 3. Create the numeric encoding map for your visualization/ML engine
sentiment_numeric_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

# 4. Map the text sentiments back to clean numbers in a separate column
df_combined["sentiment_code"] = df_combined["sentiment_text"].map(sentiment_numeric_map)

print(df_combined["sentiment_text"].value_counts())


sentiment_text
negative    12608
positive     9591
neutral      1528
Name: count, dtype: int64


In [28]:
display(df_combined.head(5))

,source_app,review_text,star_rating,recorded_at,app_version,sentiment_text,sentiment_code,normalized_text,source_row_id
0,FriMi_FinTech,Still locked out of the app after 3 days! 😩 I'...,1,2026-06-04 08:40:15,3.05.08,negative,0,still locked out of the app after 3 days! 😩 i'...,1
1,FriMi_FinTech,"Can't create new account. Showing ""run time er...",1,2026-05-30 20:01:25,NaN,negative,0,"can't create new account. showing ""run time er...",2
2,FriMi_FinTech,"First impression ""A big Run time ERROR""",2,2026-05-25 22:40:48,3.05.08,negative,0,"first impression ""a big run time error""",3
3,FriMi_FinTech,poor customer hotline service in an emergency ...,1,2026-05-24 22:05:56,3.05.07,negative,0,poor customer hotline service in an emergency ...,4
4,FriMi_FinTech,I cant create an account. I tried different de...,4,2026-05-23 19:09:12,3.05.07,positive,2,i cant create an account. i tried different de...,5


In [31]:
df_combined.to_csv(
    "data/processed/scraped_reviews_combined.csv",
    index=False,
    encoding="utf-8-sig"
)